# Recon 3
Internal 8012 service, k8s API, mounts, s3.

In [ ]:
import subprocess
cmds = [
    "cat /home/connect/.gitconfig 2>&1; echo; cat /home/connect/.bashrc 2>&1 | head -30",
    "cat /cloud/persistent-state/last-written-source-version 2>&1; ls -la /mnt/dynamic-mounts/ 2>&1; ls -la /mnt/dynamic-mounts/rootfs-* 2>&1 | head",
    "find /mnt/dynamic-mounts -maxdepth 4 2>&1 | head -60",
    "ls -la /launch-connect.sh /symlink-quarto.sh /opt/connect 2>&1; cat /launch-connect.sh 2>&1 | head -40",
    "env | grep -iE 'token|key|secret|password|cred|bucket|role' ",
]
for c in cmds:
    print(f"$ {c}")
    p = subprocess.run(c, shell=True, capture_output=True, text=True, timeout=30)
    print(p.stdout[:5000], p.stderr[:2000])
    print("---")

In [ ]:
import subprocess
script = r'''
import urllib.request, ssl, json
ctx = ssl._create_unverified_context()
def req(url, data=None):
    try:
        r = urllib.request.urlopen(url, data=data, timeout=6, context=ctx)
        body = r.read(3000)
        print(url, "->", r.status, r.headers.get("content-type"), body[:1500])
    except Exception as e:
        print(url, "FAIL:", e)
req("https://10.96.0.1/version")
req("https://10.96.0.1/api")
req("https://10.96.0.1/api/v1/namespaces")
req("https://10.96.0.1/api/v1/pods")
req("https://10.96.0.1/healthz")
req("https://10.96.0.1/openapi/v2")
'''
p = subprocess.run(["python3", "-c", script], capture_output=True, text=True, timeout=90)
print(p.stdout[:9000], p.stderr[:3000])

In [ ]:
import subprocess
script = r'''
import urllib.request
def req(url, data=None, extra_headers={}):
    try:
        h = {"User-Agent": "Mozilla/5.0", "Host": "example.com"}
        r = urllib.request.Request(url, data=data, headers={**h, **extra_headers})
        resp = urllib.request.urlopen(r, timeout=6)
        print("###", url, "->", resp.status, resp.headers.get("content-type"))
        print(resp.read(2500)[:2000])
    except Exception as e:
        print("###", url, "FAIL:", e)
# the internal queue-serving service
for p in ["/", "/__/health", "/health", "/metrics", "/queue", "/status", "/api", "/v1", "/__/auth", "/favicon.ico", "/app", "/__sockjs__"]:
    req("http://127.0.0.1:8012" + p)
req("https://127.0.0.1:8112/", None)
'''
p = subprocess.run(["python3", "-c", script], capture_output=True, text=True, timeout=90)
print(p.stdout[:12000], p.stderr[:3000])

In [ ]:
import subprocess
script = r'''
import urllib.request
def req(url):
    try:
        resp = urllib.request.urlopen(urllib.request.Request(url, headers={"User-Agent":"curl/8"}), timeout=6)
        print(url, "->", resp.status, resp.read(1000)[:700])
    except Exception as e:
        print(url, "FAIL:", e)
req("https://s3.us-east-2.amazonaws.com/vivid-production-bundle-transfer-temp/")
req("https://s3.amazonaws.com/vivid-production-bundle-transfer-temp/")
req("http://metadata.google.internal/")
req("http://169.254.170.2/v2/credentials")
'''
p = subprocess.run(["python3", "-c", script], capture_output=True, text=True, timeout=90)
print(p.stdout[:8000], p.stderr[:3000])
# ARP table equivalent: scan pod CIDR for live hosts on port 3838/8012
script2 = r'''
import socket
import concurrent.futures
def chk(ip):
    for port in [3838, 8012, 9090]:
        try:
            s = socket.create_connection((ip, port), timeout=0.4); s.close(); return (ip, port)
        except Exception:
            pass
    return None
with concurrent.futures.ThreadPoolExecutor(60) as ex:
    res = ex.map(chk, [f"192.168.4.{i}" for i in range(1,255)])
    for r in res:
        if r: print("OPEN", r)
'''
p2 = subprocess.run(["python3", "-c", script2], capture_output=True, text=True, timeout=120)
print(p2.stdout[:5000], p2.stderr[:2000])